In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

np.random.seed(0)
N=3000

x1=np.random.uniform(-2,2,N)
x2=np.random.uniform(-2,2,N)

X=np.column_stack((x1,x2))
y=np.zeros(N)

for i in range(N):
    if x1[i]**2+x2[i]**2>1.5:
        y[i]=1
    else:
        y[i]=0

y=y.reshape(-1, 1)


In [ ]:
X_train,X_temp,y_train,y_temp = train_test_split(
    X, y, test_size=0.3,random_state=42, shuffle=True
)

# Second split: 15% val, 15% test
X_val,X_test,y_val,y_test=train_test_split(
    X_temp,y_temp,test_size=0.5,random_state=42,shuffle=True
)

print("Train size:", X_train.shape[0])
print("Val size:", X_val.shape[0])
print("Test size:", X_test.shape[0])


In [ ]:
def sigmoid(z):
    return 1/(1+np.exp(-z))

def sigmoid_derivative(a):
    return a*(1-a)

def relu(z):
    return np.maximum(0,z)

def relu_derivative(z):
    dz=np.zeros_like(z)
    dz[z>0] = 1
    return dz

In [ ]:
def compute_loss(y_true,y_pred):
    eps=1e-8
    y_pred=np.clip(y_pred,eps,1-eps)
    loss= -np.mean(y_true*np.log(y_pred) + (1-y_true)*np.log(1-y_pred))
    return loss

def compute_accuracy(y_true,y_pred):
    y_bin=(y_pred>=0.5).astype(int)
    return np.mean(y_bin==y_true)

In [ ]:
def initialize_network(layer_sizes):
    weights=[]
    biases=[]

    for i in range(len(layer_sizes)-1):
        W=np.random.randn(layer_sizes[i],layer_sizes[i+1])*0.1
        b=np.zeros((1,layer_sizes[i+1]))

        weights.append(W)
        biases.append(b)

    return weights, biases

In [ ]:
def forward(X,weights,biases,activation_name):

    activations=[X]
    zs =[]

    for i in range(len(weights)-1):

        z=activations[-1] @ weights[i]+biases[i]
        zs.append(z)

        if activation_name=="sigmoid":
            a=sigmoid(z)
        else:
            a=relu(z)

        activations.append(a)

    # output layer always sigmoid
    z=activations[-1] @ weights[-1]+biases[-1]
    zs.append(z)

    a=sigmoid(z)
    activations.append(a)

    return activations,zs

In [ ]:
def backward(y,activations,zs,weights,activation_name):

    grads_W=[]
    grads_b=[]

    m=y.shape[0]

    dz=activations[-1]-y

    for i in reversed(range(len(weights))):

        a_prev=activations[i]

        dW=(a_prev.T @ dz)/m
        db=np.sum(dz,axis=0,keepdims=True)/m

        grads_W.insert(0,dW)
        grads_b.insert(0,db)

        if i!= 0:
            da_prev=dz @ weights[i].T

            if activation_name =="sigmoid":
                dz=da_prev*sigmoid_derivative(activations[i])
            else:
                dz=da_prev*relu_derivative(zs[i-1])

    return grads_W,grads_b

In [ ]:
def train_model(layer_sizes,activation_name,optimizer_name,lr=0.1,epochs=100):

    weights,biases=initialize_network(layer_sizes)

    velocity_W=[np.zeros_like(W) for W in weights]
    velocity_b=[np.zeros_like(b) for b in biases]

    beta=0.9

    train_loss_list=[]
    val_loss_list=[]
    train_acc_list=[]
    val_acc_list=[]

    grad_first=[]
    grad_last=[]

    for epoch in range(epochs):

        # forward
        activations,zs=forward(X_train,weights,biases,activation_name)

        # loss
        train_loss=compute_loss(y_train,activations[-1])
        train_acc=compute_accuracy(y_train,activations[-1])

        # backward
        grads_W,grads_b=backward(y_train,activations,zs,weights, activation_name)

        grad_first.append(np.linalg.norm(grads_W[0]))
        grad_last.append(np.linalg.norm(grads_W[-1]))

        # update
        for i in range(len(weights)):

            if optimizer_name=="sgd":
                weights[i]-=lr*grads_W[i]
                biases[i]-=lr*grads_b[i]

            else:  # momentum
                velocity_W[i]=beta*velocity_W[i]+lr*grads_W[i]
                velocity_b[i]=beta*velocity_b[i]+lr*grads_b[i]

                weights[i]-=velocity_W[i]
                biases[i]-=velocity_b[i]

        # validation
        val_act, _ = forward(X_val,weights,biases,activation_name)

        val_loss=compute_loss(y_val,val_act[-1])
        val_acc=compute_accuracy(y_val,val_act[-1])

        train_loss_list.append(train_loss)
        val_loss_list.append(val_loss)
        train_acc_list.append(train_acc)
        val_acc_list.append(val_acc)

        if epoch%10==0:
            print("Epoch:",epoch,"Train Loss:", train_loss,"Val Loss:", val_loss)

    return (weights, biases,train_loss_list,val_loss_list,train_acc_list,val_acc_list,grad_first,grad_last)

In [ ]:
layer_sizes=[2,8,8,8,1]

results=train_model(layer_sizes,activation_name="sigmoid",optimizer_name="sgd",lr=0.1,epochs=100)

(weights, biases,train_loss,val_loss,train_acc,val_acc,grad_first, grad_last)=results

plt.figure()
plt.plot(train_loss)
plt.plot(val_loss)
plt.title("Loss vs Epoch")
plt.legend(["Train","Validation"])
plt.show()

plt.figure()
plt.plot(train_acc)
plt.plot(val_acc)
plt.title("Accuracy vs Epoch")
plt.legend(["Train","Validation"])
plt.show()

test_act, _ = forward(X_test, weights, biases,"sigmoid")

print("Final Test Loss:",compute_loss(y_test, test_act[-1]))

print("Final Test Accuracy:",compute_accuracy(y_test, test_act[-1]))

plt.figure()
plt.plot(grad_first)
plt.plot(grad_last)
plt.legend(["First Layer", "Last Layer"])
plt.title("Gradient Norm Comparison")
plt.show()